In [ ]:
# ============================================================
# RANDOM FOREST CLASSIFICATION — LEAKAGE RISK REGIME
# ============================================================

# ----------------------------
# 1. IMPORT LIBRARIES
# ----------------------------
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score


# ----------------------------
# 2. CREATE OUTPUT FOLDER
# ----------------------------
os.makedirs("figures", exist_ok=True)


# ----------------------------
# 3. LOAD DATA
# ----------------------------
# Replace with your real Excel file name
file_path = "example_dataset.csv"

df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ----------------------------
# 4. CHECK CLASS DISTRIBUTION
# ----------------------------
target_col = "Leakage risk regime"

counts = df[target_col].value_counts()
proportions = df[target_col].value_counts(normalize=True)

print("\n=== Class Counts ===")
print(counts)

print("\n=== Class Proportions ===")
print(proportions)


# ----------------------------
# 5. PLOT CLASS PROPORTIONS
# ----------------------------
class_order = ["low_risk", "high_risk", "extreme"]

plt.figure(figsize=(6, 4))
proportions.reindex(class_order).plot(
    kind="bar",
    color=["#FFD966", "#6AA84F", "#CC0000"]
)

plt.title("Proportion of Leakage Risk Regimes")
plt.ylabel("Fraction of Cases")
plt.xlabel("Leakage Risk Regime")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    "figures/Figure_Proportion_LeakageRiskRegimes.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ----------------------------
# 6. SELECT INPUT FEATURES
# ----------------------------
x_cols = [
    "PERM_permeability",
    "PERM_perm_factor_of_failed_well",
    "PERM_dist_to_well_in_grid",
    "PERM_caprock_perm",
    "porosity",
    "aquifer_porosity",
    "aquifer_permeability"
]

X = df[x_cols].copy()
y = df[target_col].copy()


# ----------------------------
# 7. ENCODE TARGET LABELS
# ----------------------------
lab = LabelEncoder()
y_encoded = lab.fit_transform(y)

print("\nEncoded classes:")
for i, cls in enumerate(lab.classes_):
    print(f"{cls} -> {i}")


# ----------------------------
# 8. TRAIN / TEST SPLIT
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain size:", len(y_train))
print("Test size:", len(y_test))


# ----------------------------
# 9. TRAIN RANDOM FOREST
# ----------------------------
rf = RandomForestClassifier(
    n_estimators=350,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


# ----------------------------
# 10. PREDICTIONS
# ----------------------------
y_pred = rf.predict(X_test)


# ----------------------------
# 11. MODEL PERFORMANCE
# ----------------------------
acc = accuracy_score(y_test, y_pred)

print("\n=== Accuracy ===")
print(f"{acc:.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=lab.classes_))


# ----------------------------
# 12. CONFUSION MATRIX
# ----------------------------
labels = lab.classes_
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    ax=ax
)

ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Leakage Risk Classification")

plt.tight_layout()

fig.savefig(
    "figures/Figure_ConfusionMatrix_RF_Classification.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ----------------------------
# 13. FEATURE IMPORTANCE
# ----------------------------
importances = rf.feature_importances_

feat_imp = pd.DataFrame({
    "Feature": x_cols,
    "Importance": importances
}).sort_values(by="Importance", ascending=True)

print("\n=== Feature Importance ===")
print(feat_imp)

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(feat_imp["Feature"], feat_imp["Importance"])

ax.set_title("Random Forest — Feature Importance")
ax.set_xlabel("Importance Score")
ax.set_ylabel("Input Variables")

plt.tight_layout()

fig.savefig(
    "figures/Figure_RF_FeatureImportance_Classification.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()